# Three-tier Qwen quality-preserving ModernBERT router

This is notebook v3. It uses **published per-example answers and scores**
for three separated Qwen2.5 tiers:

| Tier | Candidate | Exact parameters | Quality evidence |
|---|---|---:|---|
| small | `Qwen2.5-1.5B` | 1.54B | Open LLM Leaderboard details |
| middle | `Qwen2.5-3B` | 3.09B | Open LLM Leaderboard details |
| large (~8B class) | `Qwen2.5-7B` | 7.61B | Open LLM Leaderboard details |

**No Qwen model weights are loaded or run.** The notebook downloads the
three aligned detail datasets, extracts their recorded prompt, answer,
and correctness metric, and trains only ModernBERT. The detail datasets
are auto-gated: accept access on Hugging Face and add `HF_TOKEN` to Colab
secrets before running.

Candidate latency is not taken from the detail datasets. It remains an
analytical estimate based on model size, BF16 precision, prompt length,
expected output length, and explicit hardware assumptions. Only the
ModernBERT decision path is timed.

Notebook 02's latest seed-44 run was **not successful**: it routed 22.21%
of 2,805 sealed-test prompts, gained 43 answers, lost 52, and missed the
macro-quality, routed-precision, and guarded-dataset gates. Its 42.55 ms
measured router p50 was below the 52.14 ms break-even point, but its
67.23 ms p95 was above break-even.

A numerical example of the new evidence path: if one published task has
250 aligned prompts, the notebook reads $250\times3=750$ recorded
prompt-model outcomes. If 1.5B and 7B both scored 1 on a prompt, 1.5B is
a safe replacement; if only 7B scored 1, it is unsafe.

> One passing run is still not investment-grade evidence. Report all
> random seeds, dataset-OOD runs, and the published evaluation protocol.


## 1. Set up Colab

Use a GPU runtime for ModernBERT training. Qwen weights are never loaded.
Before running, open and accept access to the three auto-gated detail
datasets linked in section 3, then save a read token as `HF_TOKEN` in
Colab secrets.


In [ ]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
llm_router = importlib.import_module("llm_router")
print(f"Router package ready from {Path(llm_router.__file__).resolve()}")


## 2. Freeze the run and published-evidence contracts

Change only `RUN_ID` between router experiments. Every split seed reuses
exactly the same immutable detail-dataset revisions and evaluation runs.
`random` asks prompt-level feasibility; `dataset_ood` holds out complete
published tasks and is a harder, separate claim.

V3 keeps at most 300 aligned prompts per task to keep Colab training
practical. With 37 non-overlapping tasks, the maximum is 11,100 prompts
and 33,300 recorded prompt-model outcomes. Tasks with fewer than 300 rows
contribute all available rows.


In [ ]:
import hashlib
import json
import shutil
from dataclasses import replace
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display
from transformers import AutoTokenizer

from llm_router.config import DEFAULT_CONFIG
from llm_router.experiment_comparison import (
    build_setup_comparison,
    choose_validation_setup,
    combine_threshold_searches,
)
from llm_router.hybrid_inference import (
    HybridModernBERTRouterRuntime,
    create_gradio_demo,
)
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    export_public_benchmark,
    make_complete_panel,
    run_public_benchmark,
    select_validation_policy,
    simulate_economics,
    split_benchmark,
)
from llm_router.qwen_evidence import (
    BINARY_METRIC_PRIORITY,
    audit_aligned_outcomes,
    published_binary_score,
    validate_qwen_tier_contract,
)
from llm_router.router_overhead import benchmark_modernbert_overhead
from llm_router.utils.training import seed_everything

RUN_SPECS = {
    "qwen25_random_seed_42": ("random", 42),
    "qwen25_random_seed_43": ("random", 43),
    "qwen25_random_seed_44": ("random", 44),
    "qwen25_dataset_ood_seed_42": ("dataset_ood", 42),
    "qwen25_dataset_ood_seed_43": ("dataset_ood", 43),
    "qwen25_dataset_ood_seed_44": ("dataset_ood", 44),
}
RUN_ID = "qwen25_random_seed_42"
SPLIT_MODE, SEED = RUN_SPECS[RUN_ID]

MAX_PROMPTS_PER_TASK = 300
EVIDENCE_SAMPLE_SEED = 20260821
EPOCHS = 8
MINIMUM_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 2
MEASURE_ROUTER_OVERHEAD = True
LAUNCH_INTERACTIVE_DEMO = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Choose Runtime > Change runtime type > GPU."

SETUP_SPECS = {
    "hybrid_r4": {
        "lora_r": 4,
        "lora_alpha": 8,
        "oracle_auxiliary_weight": 0.25,
        "dataset_balanced_sampling": False,
    },
    "safety_only_r4": {
        "lora_r": 4,
        "lora_alpha": 8,
        "oracle_auxiliary_weight": 0.0,
        "dataset_balanced_sampling": False,
    },
    "hybrid_r8": {
        "lora_r": 8,
        "lora_alpha": 16,
        "oracle_auxiliary_weight": 0.25,
        "dataset_balanced_sampling": False,
    },
}

CANDIDATES = {
    "Qwen2.5-1.5B": {
        "model_repo": "Qwen/Qwen2.5-1.5B-Instruct",
        "model_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
        "parameters_billions": 1.54,
        "evidence_repo": "open-llm-leaderboard/Qwen__Qwen2.5-1.5B-Instruct-details",
        "evidence_revision": "801e825efda393dafd601fc9f9fa85646050f3b5",
        "evaluation_run": "2024-09-19T16-22-58.240552",
    },
    "Qwen2.5-3B": {
        "model_repo": "Qwen/Qwen2.5-3B-Instruct",
        "model_revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
        "parameters_billions": 3.09,
        "evidence_repo": "open-llm-leaderboard/Qwen__Qwen2.5-3B-Instruct-details",
        "evidence_revision": "f5b005e407b8546e16de9349a82bbd40b0133abe",
        "evaluation_run": "2024-09-19T10-05-30.339220",
    },
    "Qwen2.5-7B": {
        "model_repo": "Qwen/Qwen2.5-7B-Instruct",
        "model_revision": "a09a35458c702b33eeacc393d103063234e8bc28",
        "parameters_billions": 7.61,
        "evidence_repo": "open-llm-leaderboard/Qwen__Qwen2.5-7B-Instruct-details",
        "evidence_revision": "98dea3336d741d4433976eeaf9e5125f39c45d5b",
        "evaluation_run": "2024-09-19T16-21-01.446061",
    },
}
validate_qwen_tier_contract(CANDIDATES)
SELECTED_MODELS = tuple(CANDIDATES)
EXCLUDED_OVERLAPPING_TASKS = {
    "leaderboard_gpqa_main",
    "leaderboard_gpqa_extended",
}

try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        "Add an accepted Hugging Face read token named HF_TOKEN to Colab secrets."
    )

evidence_contract = {
    "source": "Hugging Face Open LLM Leaderboard per-example detail datasets",
    "candidates": CANDIDATES,
    "max_prompts_per_task": MAX_PROMPTS_PER_TASK,
    "sampling_seed": EVIDENCE_SAMPLE_SEED,
    "excluded_overlapping_tasks": sorted(EXCLUDED_OVERLAPPING_TASKS),
    "correctness_policy": {
        "source": "published task-aware per-example metric",
        "metric_priority": BINARY_METRIC_PRIORITY,
        "required_values": [0, 1],
        "quality_formula": "correct / outcomes",
    },
    "candidate_latency": "analytical BF16; published runtime is unused",
}
EVIDENCE_TAG = hashlib.sha256(
    json.dumps(evidence_contract, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
OUTPUT_DIR = PROJECT_ROOT / "reports_benchmark" / RUN_ID

def log_stage(stage, **values):
    timestamp = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    details = " | ".join(f"{key}={value}" for key, value in values.items())
    print(f"[{timestamp}] {stage}" + (f" | {details}" if details else ""))

seed_everything(SEED)
log_stage(
    "contracts frozen",
    run_id=RUN_ID,
    split=SPLIT_MODE,
    evidence_tag=EVIDENCE_TAG,
    gpu=torch.cuda.get_device_name(0),
)
display(pd.DataFrame(CANDIDATES).T)


## 3. Download the published aligned Qwen answer datasets

Accept access to these auto-gated Hugging Face datasets before running:

- `open-llm-leaderboard/Qwen__Qwen2.5-1.5B-Instruct-details`
- `open-llm-leaderboard/Qwen__Qwen2.5-3B-Instruct-details`
- `open-llm-leaderboard/Qwen__Qwen2.5-7B-Instruct-details`

Each repository contains the same 39 evaluation tasks. V3 selects the
pinned evaluation run from each repository and excludes GPQA main and
extended because they overlap with GPQA Diamond. The remaining published
tasks include BBH, GPQA Diamond, IFEval, MATH-Hard categories, MMLU-Pro,
and MuSR.

The loader keeps the published rendered prompt, filtered response,
per-example metric, document hash, prompt hash, task name, and document
ID. It downloads JSONL evidence only—never Qwen weights. Each selected
metric must be exactly 0 (incorrect) or 1 (correct); a fractional or
ambiguous metric stops the run instead of silently becoming a label.


In [ ]:
def first_string(value):
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for nested in value.values():
            found = first_string(nested)
            if found is not None:
                return found
    if isinstance(value, (list, tuple)):
        for nested in value:
            found = first_string(nested)
            if found is not None:
                return found
    return None

def selected_sample_files(spec):
    files = HfApi().list_repo_files(
        spec["evidence_repo"],
        repo_type="dataset",
        revision=spec["evidence_revision"],
        token=HF_TOKEN,
    )
    suffix = f"_{spec['evaluation_run']}.jsonl"
    selected = []
    for filename in files:
        basename = Path(filename).name
        if not basename.startswith("samples_") or not basename.endswith(suffix):
            continue
        task = basename[len("samples_") : -len(suffix)]
        if task not in EXCLUDED_OVERLAPPING_TASKS:
            selected.append((task, filename))
    return sorted(selected)

raw_rows = []
published_run_metadata = {}
task_sets = {}
for model_name, spec in CANDIDATES.items():
    task_files = selected_sample_files(spec)
    task_sets[model_name] = {task for task, _ in task_files}
    results_filename = str(
        Path(task_files[0][1]).parent
        / f"results_{spec['evaluation_run']}.json"
    )
    results_path = hf_hub_download(
        repo_id=spec["evidence_repo"],
        filename=results_filename,
        repo_type="dataset",
        revision=spec["evidence_revision"],
        token=HF_TOKEN,
    )
    with open(results_path, encoding="utf-8") as handle:
        published_run_metadata[model_name] = json.load(handle)
    log_stage("published evidence download started", model=model_name, tasks=len(task_files))
    for task, filename in task_files:
        local_path = hf_hub_download(
            repo_id=spec["evidence_repo"],
            filename=filename,
            repo_type="dataset",
            revision=spec["evidence_revision"],
            token=HF_TOKEN,
        )
        with open(local_path, encoding="utf-8") as handle:
            for line in handle:
                payload = json.loads(line)
                metric_name, score = published_binary_score(payload)
                prompt = first_string(payload.get("arguments"))
                if not prompt:
                    prompt = first_string(payload.get("doc"))
                if not prompt:
                    prompt = json.dumps(
                        payload.get("doc"), ensure_ascii=False, sort_keys=True
                    )
                response = first_string(payload.get("filtered_resps"))
                if response is None:
                    response = first_string(payload.get("resps")) or ""
                document_identity = str(payload.get("doc_hash") or "")
                if not document_identity:
                    document_identity = hashlib.sha256(
                        json.dumps(
                            payload.get("doc"),
                            ensure_ascii=False,
                            sort_keys=True,
                        ).encode("utf-8")
                    ).hexdigest()
                raw_rows.append(
                    {
                        "key": f"{task}::{payload['doc_id']}",
                        "task": task,
                        "doc_id": str(payload["doc_id"]),
                        "doc_hash": document_identity,
                        "prompt_hash": str(payload.get("prompt_hash", "")),
                        "target_hash": str(payload.get("target_hash", "")),
                        "prompt": prompt,
                        "prediction": response,
                        "target": json.dumps(
                            payload.get("target"), ensure_ascii=False, sort_keys=True
                        ),
                        "score": score,
                        "published_metric": metric_name,
                        "model": model_name,
                        "source_file": filename,
                        "evidence_repo": spec["evidence_repo"],
                        "evidence_revision": spec["evidence_revision"],
                    }
                )
    log_stage("published evidence loaded", model=model_name, rows=len(raw_rows))

assert len({frozenset(tasks) for tasks in task_sets.values()}) == 1, (
    "The pinned candidate evidence repositories do not expose identical task sets."
)
raw_records = pd.DataFrame(raw_rows)
assert raw_records.groupby(["key", "model"]).size().eq(1).all()
display(
    pd.DataFrame(
        {"model": task_sets.keys(), "published_tasks": map(len, task_sets.values())}
    )
)


## 4. Align, sample, and audit the published outcomes

V3 retains only prompt keys present for all three models. It requires
document hashes and rendered prompts to agree across candidates. Sampling
is deterministic within each published task, and duplicate document hashes
are kept once globally so dataset-OOD splits cannot leak the same question
through two task variants.

Correctness comes from each benchmark's published task-aware grader. The
notebook does not try to re-grade heterogeneous tasks with a fragile text
comparison. It verifies that every score is binary, records it as the
boolean `is_correct`, and computes quality as `correct / outcomes`. For
example, 240 correct rows out of 300 give quality $240/300=80\%$.

The notebook downloads the pinned Qwen tokenizer only to count input and
output tokens. Tokenization is not model inference. Realized completion
length remains excluded from analytical latency.


In [ ]:
model_count_by_key = raw_records.groupby("key").model.nunique()
complete_keys = set(model_count_by_key[model_count_by_key.eq(len(CANDIDATES))].index)
aligned = raw_records.loc[raw_records.key.isin(complete_keys)].copy()
assert not aligned.empty
assert aligned.groupby("key").doc_hash.nunique().le(1).all()
prompt_variants = aligned.groupby("key").prompt.nunique()
if not prompt_variants.le(1).all():
    raise ValueError(
        f"{int((prompt_variants > 1).sum())} keys have model-dependent prompts."
    )

canonical = (
    aligned.sort_values(["key", "model"])
    .drop_duplicates("key")
    [["key", "task", "doc_hash", "prompt"]]
)
selected_keys = []
seen_doc_hashes = set()
for task_index, (task, task_rows) in enumerate(
    canonical.groupby("task", sort=True), start=1
):
    task_rows = task_rows.loc[~task_rows.doc_hash.isin(seen_doc_hashes)]
    count = min(MAX_PROMPTS_PER_TASK, len(task_rows))
    rng = np.random.default_rng(EVIDENCE_SAMPLE_SEED + task_index)
    positions = np.sort(rng.choice(len(task_rows), size=count, replace=False))
    selected = task_rows.iloc[positions]
    selected_keys.extend(selected.key)
    seen_doc_hashes.update(selected.doc_hash)

records = aligned.loc[aligned.key.isin(selected_keys)].copy()
records["example_id"] = records.key
records["dataset"] = records.task.str.removeprefix("leaderboard_")
records["source_split"] = "published_evaluation"
records["recorded_cost"] = 0.0

tokenizer = AutoTokenizer.from_pretrained(
    CANDIDATES["Qwen2.5-7B"]["model_repo"],
    revision=CANDIDATES["Qwen2.5-7B"]["model_revision"],
    use_fast=True,
)
prompt_token_counts = {
    key: len(tokenizer.encode(prompt, add_special_tokens=False))
    for key, prompt in records.drop_duplicates("key")[["key", "prompt"]].itertuples(
        index=False, name=None
    )
}
records["prompt_tokens"] = records.key.map(prompt_token_counts).astype(int)
records["completion_tokens"] = [
    len(tokenizer.encode(str(value), add_special_tokens=False))
    for value in records.prediction
]
del tokenizer

expected_rows = len(selected_keys) * len(CANDIDATES)
assert len(records) == expected_rows
records, quality_audit = audit_aligned_outcomes(records, SELECTED_MODELS)
assert records.groupby("example_id").model.nunique().eq(len(CANDIDATES)).all()
assert records.score.eq(records.is_correct.astype(float)).all()
log_stage(
    "published evidence aligned",
    prompts=len(selected_keys),
    outcomes=len(records),
    tasks=records.dataset.nunique(),
    qwen_weights_loaded=False,
)
display(
    records.pivot_table(
        index="dataset", columns="model", values="score", aggfunc="mean"
    ).style.format("{:.1%}")
)
display(quality_audit)


## 5. Attach analytical latency and audit leakage

The panel rejects missing prompt/model pairs and duplicate content crossing
splits. Candidate latency is computed from parameter count, BF16 weight
precision, prompt size, expected output length, and dated hardware
assumptions. Published runtime and realized completion length are excluded.

For active parameters $P$, effective compute $F$, bandwidth $B$, and
precision $b$, the core terms are:

$$t_{compute/token}=\frac{2P}{F},\qquad
t_{memory/pass}=\frac{Pb/8}{B}.$$

The code multiplies every recorded completion length by 100 and requires
analytical latency to remain identical. That is the leakage check.


In [ ]:
analytical_records = records.assign(
    formatted_prompt=records.prompt,
    ground_truth=records.target,
)
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        parameters_billions=spec["parameters_billions"],
        active_parameters_billions=spec["parameters_billions"],
        architecture="autoregressive",
        weight_bits=16,
    )
    for name, spec in CANDIDATES.items()
)
scenario = EconomicsScenario(
    name="qwen25-three-tier-analytical-latency-poc",
    as_of="2026-08-21",
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="Published Qwen answer evidence; analytical BF16 candidate latency only.",
)
simulated = simulate_economics(analytical_records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)

counterfactual = analytical_records.copy()
counterfactual["completion_tokens"] *= 100
counterfactual_simulated = simulate_economics(counterfactual, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual_simulated.simulated_latency_s,
), "Leakage check failed: completion length changed analytical latency."

split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_indices = {"train": split.train, "validation": split.validation, "test": split.test}
prompt_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in split_indices.items()
}
assert prompt_hashes["train"].isdisjoint(prompt_hashes["validation"])
assert prompt_hashes["train"].isdisjoint(prompt_hashes["test"])
assert prompt_hashes["validation"].isdisjoint(prompt_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)

quality_summary = records.groupby("model").is_correct.mean().rename("quality")
latency_summary = simulated.groupby("model").simulated_latency_s.mean().rename(
    "mean_analytical_latency_s"
)
panel_summary = pd.concat([quality_summary, latency_summary], axis=1)
panel_summary["latency_vs_7b"] = (
    panel_summary.mean_analytical_latency_s
    / panel_summary.loc["Qwen2.5-7B", "mean_analytical_latency_s"]
)
display(panel_summary.style.format("{:.2%}", subset=["quality", "latency_vs_7b"]))
display(
    pd.DataFrame(
        {
            "split": list(split_indices),
            "prompts": [len(indices) for indices in split_indices.values()],
        }
    )
)
log_stage(
    "Leakage check and split audit passed",
    complete_prompts=len(panel.examples),
    train=len(split.train),
    validation=len(split.validation),
    test=len(split.test),
)


## 6. Verify validation-only oracle headroom and scenario sensitivity

The hindsight oracle can see recorded outcomes and chooses the fastest
candidate that preserves the training-selected fallback's quality. It is
training-only. If it cannot save latency, no learned prompt-only router can
rescue the panel under this objective.

Example: if 1.5B and 7B both score 1, the oracle chooses 1.5B. If only 7B
scores 1, it chooses 7B. V3 requires positive validation oracle savings in
every declared analytical scenario before spending time on ModernBERT.


In [ ]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    choices = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows_index = np.arange(len(indices))
    selected = choices[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows_index, selected].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows_index, selected].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(selected == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for scenario_name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=scenario_name, **overrides)
    variant_records = simulate_economics(analytical_records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    sensitivity_rows.append(
        {"scenario": scenario_name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
assert (sensitivity.latency_savings > 0).all(), (
    "No robust oracle headroom. Stop: this panel cannot support the routing claim."
)
display(sensitivity.style.format({"quality_retention": "{:.2%}", "latency_savings": "{:.2%}"}))


## 7. Understand the objective, loss, and calibration

ModernBERT predicts one replacement-safety probability for each
non-fallback candidate:

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

With default $\epsilon_q=0$, matching the fallback is safe. Independent,
class-balanced binary cross-entropy trains the deployed safety heads. A
training-only hindsight-oracle head adds 0.25 times its quality-first
latency-regret loss in the hybrid setups:

$$\mathcal L_{train}=\mathcal L_{safety}+0.25\mathcal L_{oracle}.$$

Platt scalers are fitted per candidate. Threshold search uses out-of-fold
validation probabilities, so a row never calibrates its own confidence.
The sealed test remains unopened until one setup and a contiguous block of
at least two feasible thresholds are frozen.

Numerical example: if the small candidate is safe on 30 of 100 training
prompts, its positive BCE weight is $70/30=2.33$. A safe example predicted
at 0.8 contributes $2.33[-\log(0.8)]\approx0.52$ before averaging.


## 8. Train every declared router setup

The three candidate tiers are fixed evidence. The validation comparison
asks whether the oracle auxiliary loss helps and whether rank-8 LoRA adds
useful router capacity. Test outcomes do not select among these setups.


In [ ]:
def make_epoch_logger(setup_name):
    def report(row):
        marker = "BEST" if row["is_best_epoch"] else "    "
        stop = " | early-stop" if row["will_stop_early"] else ""
        print(
            f"[{setup_name}] epoch={int(row['epoch'])}/{EPOCHS} {marker} "
            f"train={row['train_total_loss']:.4f} "
            f"validation={row['validation_total_loss']:.4f} "
            f"best_epoch={int(row['best_epoch_so_far'])} "
            f"seconds={row['epoch_seconds']:.1f} "
            f"examples_per_second={row['train_examples_per_second']:.1f}"
            f"{stop}"
        )
    return report

trainings = {}
setup_configs = {}
for setup_name, spec in SETUP_SPECS.items():
    setup_config = replace(
        DEFAULT_CONFIG,
        seed=SEED,
        lora_r=spec["lora_r"],
        lora_alpha=spec["lora_alpha"],
    )
    setup_configs[setup_name] = setup_config
    log_stage("router training started", setup=setup_name, **spec)
    training = train_modernbert_hybrid_poc(
        panel,
        split,
        config=setup_config,
        epochs=EPOCHS,
        batch_size=8,
        learning_rate=1e-4,
        head_learning_rate=2e-4,
        minimum_epochs=MINIMUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        quality_epsilon=0.0,
        safety_loss_weight=1.0,
        oracle_auxiliary_weight=spec["oracle_auxiliary_weight"],
        dataset_balanced_sampling=spec["dataset_balanced_sampling"],
        device=DEVICE,
        progress_callback=make_epoch_logger(setup_name),
    )
    training.model.to("cpu")
    torch.cuda.empty_cache()
    trainings[setup_name] = training
    log_stage(
        "router training finished",
        setup=setup_name,
        best_epoch=training.best_epoch,
        epochs=training.epochs_completed,
        truncation=f"{training.input_diagnostics['truncation_rate']:.2%}",
    )


## 9. Compare setups on validation and freeze one policy

Activation requires aggregate and macro quality bounds, harm-rate control,
routed safety precision, the guarded-dataset floor when enough examples
exist, positive savings at both 4 ms and 20 ms overhead, and at least two
adjacent passing thresholds. An isolated lucky threshold fails closed.


In [ ]:
policy_kwargs = {
    "objective": "latency",
    "minimum_quality_retention": DEFAULT_CONFIG.minimum_quality_retention,
    "confidence": DEFAULT_CONFIG.quality_confidence,
    "validation_quality_margin": DEFAULT_CONFIG.validation_quality_margin,
    "minimum_predicted_savings": DEFAULT_CONFIG.minimum_predicted_speedup,
    "router_overhead_s": scenario.router_overhead_s,
    "conservative_router_overhead_s": DEFAULT_CONFIG.conservative_router_overhead_s,
    "minimum_macro_quality_retention": DEFAULT_CONFIG.minimum_macro_quality_retention,
    "maximum_quality_loss_rate_ucl": DEFAULT_CONFIG.maximum_quality_loss_rate_ucl,
    "minimum_routed_safety_precision_lcb": DEFAULT_CONFIG.minimum_routed_safety_precision_lcb,
    "minimum_guarded_dataset_quality_retention_lcb": (
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    "minimum_guarded_dataset_prompts": DEFAULT_CONFIG.minimum_guarded_dataset_prompts,
    "minimum_consecutive_feasible_thresholds": (
        DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
    ),
    "seed": SEED,
}
selections = {}
for setup_name, training in trainings.items():
    selection = select_validation_policy(
        panel,
        split,
        routing_probabilities=training.safety_probabilities,
        router_name=f"modernbert_qwen_tier_router__{setup_name}",
        **policy_kwargs,
    )
    selections[setup_name] = selection
    log_stage(
        "validation policy evaluated",
        setup=setup_name,
        active=selection.router_active,
        threshold=selection.selected_threshold,
        reason="; ".join(selection.failure_reasons) or "all gates passed",
    )

setup_comparison = build_setup_comparison(trainings, selections)
setup_threshold_search = combine_threshold_searches(selections)
display(
    setup_comparison.sort_values(
        ["router_active", "conservative_resource_savings"], ascending=False
    ).style.format(
        {
            "quality_retention_lcb": "{:.2%}",
            "routed_safety_precision_lcb": "{:.2%}",
            "safe_opportunity_recall": "{:.2%}",
            "routed_fraction": "{:.2%}",
            "nominal_resource_savings": "{:.2%}",
            "conservative_resource_savings": "{:.2%}",
        }
    )
)

SELECTED_SETUP = choose_validation_setup(setup_comparison)
setup_comparison["selected_for_test"] = setup_comparison.setup.eq(SELECTED_SETUP)
selected_training = trainings[SELECTED_SETUP]
selected_config = setup_configs[SELECTED_SETUP]
selected_spec = SETUP_SPECS[SELECTED_SETUP]
frozen_validation_policy = selections[SELECTED_SETUP]
log_stage(
    "POLICY FROZEN — sealed test may now open",
    setup=SELECTED_SETUP,
    active=frozen_validation_policy.router_active,
    threshold=frozen_validation_policy.selected_threshold,
    feasible_block=int(
        setup_comparison.set_index("setup").loc[
            SELECTED_SETUP, "feasible_block_size"
        ]
    ),
)


## 10. Measure ModernBERT overhead only

This measures tokenization, host-to-device transfer, and ModernBERT
inference on the named Colab GPU. It does not time any Qwen candidate and
cannot change the already-frozen threshold. The measured p50 and p95 are
compared with the eventual policy-specific break-even overhead.


In [ ]:
router_overhead_benchmark = None
if MEASURE_ROUTER_OVERHEAD:
    validation_examples = panel.examples.iloc[split.validation]
    router_overhead_benchmark = benchmark_modernbert_overhead(
        selected_training.model,
        selected_training.tokenizer,
        validation_examples.prompt.to_numpy(),
        validation_examples.prompt_tokens.to_numpy(),
        device=DEVICE,
        max_input_tokens=selected_config.max_input_tokens,
        timed_requests=min(100, len(validation_examples)),
        warmup_requests=10,
        seed=SEED,
    )
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    log_stage(
        "ModernBERT overhead measured",
        p50_ms=f"{measured['p50']:.2f}",
        p95_ms=f"{measured['p95']:.2f}",
        candidate_latency="analytical-only",
        policy_changed=False,
    )
    display(pd.DataFrame(router_overhead_benchmark.summary).loc[
        ["mean", "p50", "p95", "maximum"],
        ["end_to_end_ms", "model_only_ms"],
    ])


## 11. Open the sealed test exactly once

A non-trivial router passes only when every predeclared sealed-test safety
and savings gate passes. Fallback-only behavior is a safe operational
result, but it is not evidence that learned routing works.


In [ ]:
result = run_public_benchmark(
    panel,
    split,
    routing_probabilities=selected_training.safety_probabilities,
    router_name="modernbert_qwen_tier_router",
    selected_setup=SELECTED_SETUP,
    decision_metadata={
        "router_input_tokens": selected_training.router_input_lengths,
        "router_was_truncated": selected_training.router_was_truncated,
    },
    **policy_kwargs,
)
assert result.selected_threshold == frozen_validation_policy.selected_threshold
assert result.router_active == frozen_validation_policy.router_active

router_metrics = result.summary.loc["modernbert_qwen_tier_router"]
routed = result.decisions.selected_model.ne(result.fallback_model)
gained = result.decisions.quality_delta.gt(0)
lost = result.decisions.quality_delta.lt(0)
log_stage(
    "sealed test complete",
    single_run_passed=result.single_run_passed,
    routed=f"{routed.mean():.2%}",
    gained=int(gained.sum()),
    lost=int(lost.sum()),
    net_answers=int(gained.sum() - lost.sum()),
    retention_lcb=f"{router_metrics.quality_retention_lcb:.2%}",
    savings_4ms=f"{router_metrics.resource_savings:.2%}",
    savings_20ms=f"{router_metrics.conservative_resource_savings:.2%}",
)
display(result.summary)
display(result.router_overhead_sensitivity)
if result.failure_reasons:
    print("Failure reasons:")
    for reason in result.failure_reasons:
        print("-", reason)

break_even_ms = float(
    result.router_overhead_sensitivity.break_even_router_overhead_ms.iloc[0]
)
overhead_rows = [
    {"source": "frozen nominal assumption", "overhead_ms": 4.0},
    {"source": "frozen conservative assumption", "overhead_ms": 20.0},
]
if router_overhead_benchmark is not None:
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    overhead_rows.extend(
        [
            {"source": "measured ModernBERT p50", "overhead_ms": measured["p50"]},
            {"source": "measured ModernBERT p95", "overhead_ms": measured["p95"]},
        ]
    )
overhead_comparison = pd.DataFrame(overhead_rows)
overhead_comparison["break_even_ms"] = break_even_ms
overhead_comparison["below_break_even"] = (
    overhead_comparison.overhead_ms < break_even_ms
)
display(overhead_comparison)


## 12. Diagnose task mix, tier usage, and truncation

Investor-facing averages must not hide that one task supplies all gains or
another absorbs most losses. The tables below expose net answers and
conservative savings by dataset, selected-tier counts, and truncation.


In [ ]:
decisions = result.decisions.assign(
    gained=gained,
    lost=lost,
    routed=routed,
)
dataset_outcomes = decisions.groupby("dataset").agg(
    prompts=("dataset", "size"),
    routed=("routed", "sum"),
    gains=("gained", "sum"),
    losses=("lost", "sum"),
    truncated=("router_was_truncated", "sum"),
)
dataset_outcomes["net_answers"] = (
    dataset_outcomes.gains - dataset_outcomes.losses
)
dataset_outcomes["routed_fraction"] = (
    dataset_outcomes.routed / dataset_outcomes.prompts
)
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("modernbert_qwen_tier_router")
].set_index("dataset")
dataset_outcomes = dataset_outcomes.join(
    router_dataset_metrics[
        ["resource_savings", "conservative_resource_savings"]
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
net_order = dataset_outcomes.sort_values("net_answers")
axes[0].barh(
    net_order.index,
    net_order.net_answers,
    color=np.where(net_order.net_answers >= 0, "tab:green", "tab:red"),
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(title="Net correct answers by dataset", xlabel="Gains minus losses")
savings_order = dataset_outcomes.sort_values("conservative_resource_savings")
axes[1].barh(
    savings_order.index,
    100 * savings_order.conservative_resource_savings,
    color=np.where(
        savings_order.conservative_resource_savings >= 0,
        "tab:blue",
        "tab:orange",
    ),
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(title="Savings at 20 ms overhead", xlabel="Analytical savings (%)")
plt.tight_layout()
plt.show()

display(dataset_outcomes.sort_values("net_answers"))
display(
    decisions.groupby("selected_model").agg(
        prompts=("selected_model", "size"),
        gains=("gained", "sum"),
        losses=("lost", "sum"),
    )
)
display(
    decisions.groupby("router_was_truncated").agg(
        prompts=("dataset", "size"),
        routed=("routed", "sum"),
        gains=("gained", "sum"),
        losses=("lost", "sum"),
    )
)


## 13. Export the reconstructable artifact and optional demo

The ZIP contains the sampled published Qwen outcomes, correctness/quality
audit, pinned evidence
contract, setup comparison, threshold frontiers, sealed-test decisions,
ModernBERT adapter and heads, Platt parameters, analytical scenario, and
timing diagnostics. The Gradio demo loads no Qwen weights; it shows the
routing decision and analytical candidate latency.


In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
setup_comparison.to_csv(report_dir / "setup_comparison.csv", index=False)
setup_threshold_search.to_csv(
    report_dir / "setup_threshold_search.csv", index=False
)
records.to_parquet(report_dir / "qwen_candidate_records.parquet", index=False)
quality_audit.to_csv(report_dir / "qwen_quality_audit.csv", index=False)
panel_summary.to_csv(report_dir / "qwen_candidate_panel_summary.csv")
(report_dir / "qwen_evidence_contract.json").write_text(
    json.dumps(
        {**evidence_contract, "evidence_tag": EVIDENCE_TAG},
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
(report_dir / "published_evaluation_metadata.json").write_text(
    json.dumps(published_run_metadata, indent=2, sort_keys=True),
    encoding="utf-8",
)
for setup_name, training in trainings.items():
    setup_dir = report_dir / "setup_diagnostics" / setup_name
    setup_dir.mkdir(parents=True, exist_ok=True)
    training.history.to_csv(setup_dir / "training_history.csv", index=False)
    training.calibration_diagnostics.to_csv(
        setup_dir / "calibration_diagnostics.csv", index=False
    )

artifact_dir = export_modernbert_hybrid_poc(
    selected_training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.single_run_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    validation_quality_margin=DEFAULT_CONFIG.validation_quality_margin,
    minimum_macro_quality_retention=DEFAULT_CONFIG.minimum_macro_quality_retention,
    maximum_quality_loss_rate_ucl=DEFAULT_CONFIG.maximum_quality_loss_rate_ucl,
    minimum_routed_safety_precision_lcb=(
        DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    minimum_guarded_dataset_prompts=DEFAULT_CONFIG.minimum_guarded_dataset_prompts,
    conservative_router_overhead_s=DEFAULT_CONFIG.conservative_router_overhead_s,
    minimum_consecutive_feasible_thresholds=(
        DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
    ),
    benchmark_fingerprint=result.benchmark_fingerprint,
    setup_name=SELECTED_SETUP,
    oracle_auxiliary_weight=selected_spec["oracle_auxiliary_weight"],
    router_overhead_benchmark=(
        router_overhead_benchmark.summary
        if router_overhead_benchmark is not None
        else None
    ),
    config=selected_config,
)
overhead_comparison.to_csv(
    report_dir / "modernbert_overhead_comparison.csv", index=False
)
if router_overhead_benchmark is not None:
    router_overhead_benchmark.export(report_dir)

demo_runtime = HybridModernBERTRouterRuntime.from_training_result(
    selected_training,
    model_names=panel.models,
    fallback_model=result.fallback_model,
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    scenario=scenario,
    config=selected_config,
    device=DEVICE,
)
demo = create_gradio_demo(demo_runtime)
if LAUNCH_INTERACTIVE_DEMO:
    demo.launch(share=True, debug=False, prevent_thread_lock=True)

bundle_path = shutil.make_archive(str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR)
print("Reports:", report_dir)
print("Router artifact:", artifact_dir)
print("ZIP:", bundle_path)
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    pass


## 14. Final interpretation checklist

Before a LinkedIn or investor claim, confirm all of the following:

- the evidence tag, detail-dataset revisions, and evaluation run IDs are in the ZIP;
- every retained prompt has one published outcome for all three Qwen tiers;
- `qwen_quality_audit.csv` shows correct, incorrect, and quality counts;
- every `score` equals the exported boolean `is_correct`;
- no Qwen model weights were loaded or generated in this notebook;
- setup and threshold selection used validation only;
- at least two neighboring thresholds passed, or the router failed closed;
- the sealed test opened once and `single_run_passed` is explicit;
- quality-retention, macro, harm, precision, subgroup, and savings gates pass;
- ModernBERT p50 and p95 are compared with break-even overhead;
- candidate latency is described as analytical BF16, not measured latency;
- no one published task supplies all net gains and no tier has a zero route rate;
- seeds 42/43/44 are reported together without cherry-picking; and
- dataset-OOD evidence is reported separately.

A useful numerical investor summary has both sides: “the router sent X% of
prompts to smaller tiers and saved Y% analytical latency, while its 95%
quality-retention lower bound was Z%.” Never present Y without Z, and do
not turn analytical milliseconds into guaranteed dollars without target-
hardware calibration.
